# Image Enhancer ViT

Trains a compact Vision Transformer on `data/processed/train`, `data/processed/val`, `data/processed/test`, and `data/processed/labels.csv`. Input is RGB `[1, 3, 384, 384]`; output is `[brightness, contrast, saturation]` correction parameters.

In [5]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    RESAMPLE_BILINEAR = Image.Resampling.BILINEAR
except AttributeError:
    RESAMPLE_BILINEAR = Image.BILINEAR

SEED = 42
IMAGE_SIZE = 384
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
NUM_WORKERS = 0
TARGET_NAMES = ["brightness", "contrast", "saturation"]
DISTORTION_LEVELS = ["none", "small", "high"]
DISTORTION_TO_INDEX = {name: index for index, name in enumerate(DISTORTION_LEVELS)}

ROOT = Path.cwd()
if ROOT.name != "ml" and (ROOT / "ml").exists():
    ROOT = ROOT / "ml"

DATA_DIR = ROOT / "data" / "processed"
SPLIT_DIRS = {split: DATA_DIR / split for split in ("train", "val", "test")}
LABELS_CSV = DATA_DIR / "labels.csv"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Device: {DEVICE}")
print(f"Dataset: {DATA_DIR}")

Device: cpu
Dataset: c:\Users\Марсель\Documents\practice\ml\data\processed


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, tokens):
        normed = self.norm1(tokens)
        attended, _ = self.attn(normed, normed, normed, need_weights=False)
        tokens = tokens + attended
        return tokens + self.mlp(self.norm2(tokens))


class EnhancementViT(nn.Module):
    def __init__(self, patch_size=32, dim=196, depth=6, heads=4, mlp_ratio=2.0, dropout=0.3):
        super().__init__()
        if IMAGE_SIZE % patch_size != 0:
            raise ValueError("IMAGE_SIZE must be divisible by patch_size")
        self.patch = nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size)
        token_count = (IMAGE_SIZE // patch_size) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.zeros(1, token_count + 1, dim))
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[
            TransformerBlock(dim, heads, mlp_ratio=mlp_ratio, dropout=dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 3)
        nn.init.trunc_normal_(self.pos, std=0.02)
        nn.init.trunc_normal_(self.cls, std=0.02)

    def forward(self, image):
        tokens = self.patch(image).flatten(2).transpose(1, 2)
        cls = self.cls.expand(image.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos
        tokens = self.dropout(tokens)
        tokens = self.blocks(tokens)
        return self.head(self.norm(tokens[:, 0]))


model = EnhancementViT().to(DEVICE)
model(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)).shape

: 

In [ ]:
class EnhancementDataset(Dataset):
    def __init__(self, split, labels_csv=LABELS_CSV, split_dirs=SPLIT_DIRS):
        self.split = split
        self.split_dirs = {name: Path(path) for name, path in split_dirs.items()}
        with Path(labels_csv).open("r", newline="", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            required = ["image_id", "brightness", "contrast", "saturation", "distortion_level"]
            if reader.fieldnames is None or any(field not in reader.fieldnames for field in required):
                raise ValueError(f"Labels must contain {required}, got {reader.fieldnames}")
            self.rows = [row for row in reader if row["image_id"].startswith(f"{split}_")]

        if not self.rows:
            raise ValueError(f"No {split} labels found in {labels_csv}")
        missing = [row["image_id"] for row in self.rows if not (self.split_dirs[self.split] / f"{row['image_id']}.jpg").exists()]
        if missing:
            raise FileNotFoundError(f"Missing {len(missing)} {split} images, first={missing[0]}")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        image_path = self.split_dirs[self.split] / f"{row['image_id']}.jpg"
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            if image.size != (IMAGE_SIZE, IMAGE_SIZE):
                image = image.resize((IMAGE_SIZE, IMAGE_SIZE), RESAMPLE_BILINEAR)
            array = np.asarray(image, dtype=np.float32) / 255.0

        tensor = torch.from_numpy(array).permute(2, 0, 1)
        target = torch.tensor([
            float(row["brightness"]),
            float(row["contrast"]),
            float(row["saturation"]),
        ], dtype=torch.float32)
        distortion = torch.tensor(DISTORTION_TO_INDEX[row["distortion_level"]], dtype=torch.long)
        return tensor, target, distortion


train_dataset = EnhancementDataset("train")
val_dataset = EnhancementDataset("val")
test_dataset = EnhancementDataset("test")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

print(f"Samples: train={len(train_dataset)} val={len(val_dataset)} test={len(test_dataset)}")
batch = next(iter(train_loader))
batch[0].shape, batch[1].shape, batch[2].shape

: 

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_fn = nn.SmoothL1Loss(reduction="none")
    total_loss = 0.0
    total_param_loss = torch.zeros(3, device=DEVICE)
    level_loss = torch.zeros(len(DISTORTION_LEVELS), device=DEVICE)
    level_param_loss = torch.zeros(len(DISTORTION_LEVELS), 3, device=DEVICE)
    level_count = torch.zeros(len(DISTORTION_LEVELS), device=DEVICE)
    count = 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for image, target, distortion in loader:
            image = image.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True)
            distortion = distortion.to(DEVICE, non_blocking=True)
            pred = model(image)
            param_loss = loss_fn(pred, target)
            sample_loss = param_loss.mean(dim=1)
            loss = sample_loss.mean()

            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

            batch_size = image.size(0)
            total_loss += sample_loss.detach().sum().item()
            total_param_loss += param_loss.detach().sum(dim=0)
            count += batch_size

            for level_index in range(len(DISTORTION_LEVELS)):
                mask = distortion == level_index
                if mask.any():
                    level_loss[level_index] += sample_loss.detach()[mask].sum()
                    level_param_loss[level_index] += param_loss.detach()[mask].sum(dim=0)
                    level_count[level_index] += mask.sum()

    metrics = {
        "total_loss": total_loss / count,
        "param_loss": (total_param_loss / count).detach().cpu().tolist(),
        "level_loss": {},
        "level_param_loss": {},
    }
    for level_index, level_name in enumerate(DISTORTION_LEVELS):
        if level_count[level_index] == 0:
            metrics["level_loss"][level_name] = None
            metrics["level_param_loss"][level_name] = None
            continue
        metrics["level_loss"][level_name] = (level_loss[level_index] / level_count[level_index]).item()
        metrics["level_param_loss"][level_name] = (level_param_loss[level_index] / level_count[level_index]).detach().cpu().tolist()
    return metrics


def format_metrics(prefix, metrics):
    param_loss = metrics["param_loss"]
    parts = [
        f"{prefix}_total={metrics['total_loss']:.5f}",
        f"{prefix}_param=[b={param_loss[0]:.5f}, c={param_loss[1]:.5f}, s={param_loss[2]:.5f}]",
    ]
    for level in DISTORTION_LEVELS:
        level_total = metrics["level_loss"][level]
        level_param = metrics["level_param_loss"][level]
        if level_total is None:
            continue
        parts.append(
            f"{prefix}_{level}={level_total:.5f}"
            f"/[b={level_param[0]:.5f}, c={level_param[1]:.5f}, s={level_param[2]:.5f}]"
        )
    return " ".join(parts)


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_state = None
    best_val = float("inf")

    for epoch in tqdm(range(epochs), desc="Training"):
        train_metrics = run_epoch(model, train_loader, optimizer)
        val_metrics = run_epoch(model, val_loader)
        if val_metrics["total_loss"] < best_val:
            best_val = val_metrics["total_loss"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        tqdm.write(
            f"Epoch {epoch + 1:02d} "
            f"{format_metrics('train', train_metrics)} "
            f"{format_metrics('val', val_metrics)}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


model = train_model(model, train_loader, val_loader)
test_metrics = run_epoch(model, test_loader)
print(format_metrics("test", test_metrics))

: 

In [ ]:
def export_onnx(model, path):
    model.eval().cpu()
    dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)
    torch.onnx.export(
        model,
        dummy,
        path,
        input_names=["image"],
        output_names=["params"],
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )
    return path


checkpoint_path = MODEL_DIR / "enhancer.onnx"
export_onnx(model, checkpoint_path)
print(f"Exported {checkpoint_path}")
checkpoint_path

: 